# Ablation (Table 2)

Attention-rollout AOPC for each medical VLM and its base, on ROCOv2 vs COCO.
ROCOv2 keeps words tagged by the medical NER; COCO drops stopwords (no medical
entities in natural-image captions). ROCOv2 uses the first 100 samples.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import wordlevel as wl

gemma_tok = AutoTokenizer.from_pretrained("google/medgemma-1.5-4b-it")
llavamed_tok = AutoTokenizer.from_pretrained("microsoft/llava-med-v1.5-mistral-7b")
llava_tok = AutoTokenizer.from_pretrained("llava-hf/llava-1.5-7b-hf")

One cell = reconstruct, clean, take the attention AOPC.

In [ ]:
def coco_cell(csv, tok):
    words = wl.reconstruct_words(pd.read_csv(csv), tok)
    return wl.attention_aopc(wl.drop_stopwords(words))

def roco_cell(csv, tok, n=100):
    df = pd.read_csv(csv)
    df = df[df["sample_idx"] < n]
    return wl.attention_aopc(wl.clean_roco(wl.reconstruct_words(df, tok)))

### ROCOv2

In [ ]:
aopc = {}
aopc[("ROCOv2", "MedGemma")]  = roco_cell("data/per_token_drops_medgemma.csv", gemma_tok)
aopc[("ROCOv2", "Gemma 3")]   = roco_cell("data/ablation/per_token_drops_gemma_rocoo.csv", gemma_tok)
aopc[("ROCOv2", "LLaVA-Med")] = roco_cell("data/per_token_drops_llava_med.csv", llavamed_tok)
aopc[("ROCOv2", "LLaVA-1.5")] = roco_cell("data/ablation/llava_rocco_drops100.csv", llava_tok)

### COCO

In [ ]:
aopc[("COCO", "MedGemma")]  = coco_cell("data/ablation/per_token_drops_medgemma_coco.csv", gemma_tok)
aopc[("COCO", "Gemma 3")]   = coco_cell("data/ablation/per_token_drops_gemma_coco.csv", gemma_tok)
aopc[("COCO", "LLaVA-Med")] = coco_cell("data/ablation/per_token_drops_llavamed_coco.csv", llavamed_tok)
aopc[("COCO", "LLaVA-1.5")] = coco_cell("data/ablation/llava_coco_drops.csv", llava_tok)

### Table 2

In [ ]:
models = ["MedGemma", "Gemma 3", "LLaVA-Med", "LLaVA-1.5"]
pd.DataFrame({m: {ds: round(aopc[(ds, m)], 4) for ds in ["ROCOv2", "COCO"]}
              for m in models})